In [1]:
%pip install openmeteo-requests requests-cache retry-requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

eia = pd.read_csv("../data/raw/eia_energy_data.csv", parse_dates=["datetime_utc"])
print(eia.shape)
eia.head()

(17288, 7)


,datetime_utc,region,demand_mwh,hour,day_of_week,month,is_weekend
0,2026-03-15 22:00:00,BPAT,7608,22,6,3,1
1,2026-03-15 23:00:00,BPAT,7638,23,6,3,1
2,2026-03-16 00:00:00,BPAT,7784,0,0,3,0
3,2026-03-16 01:00:00,BPAT,7957,1,0,3,0
4,2026-03-16 02:00:00,BPAT,8097,2,0,3,0


In [ ]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Setup connection to Open-Meteo with caching and retry
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Region -> (city, latitude, longitude)
regions = {
    "BPAT": ("Seattle",      47.6062, -122.3321),
    "CISO": ("Sacramento",   38.5816, -121.4944),
    "ERCO": ("Dallas",       32.7767,  -96.7970),
    "ISNE": ("Boston",       42.3601,  -71.0589),
    "MISO": ("Chicago",      41.8781,  -87.6298),
    "NYIS": ("New York",     40.7128,  -74.0060),
    "PJM":  ("Philadelphia", 39.9526,  -75.1652),
    "SWPP": ("Oklahoma City",35.4676,  -97.5164),
}

START = "2025-06-20"
END   = "2026-06-20"

url = "https://archive-api.open-meteo.com/v1/archive"
all_weather = []

for region, (city, lat, lon) in regions.items():
    print(f"Fetching {city} for {region}...")

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START,
        "end_date": END,
        "hourly": "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation",
    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    hourly = response.Hourly()

    hourly_data = {
        "datetime_utc": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature":   hourly.Variables(0).ValuesAsNumpy(),
        "apparent_temp": hourly.Variables(1).ValuesAsNumpy(),
        "humidity":      hourly.Variables(2).ValuesAsNumpy(),
        "precipitation": hourly.Variables(3).ValuesAsNumpy(),
    }
    hourly_data["region"] = region

    df = pd.DataFrame(data=hourly_data)
    all_weather.append(df)

weather_df = pd.concat(all_weather, ignore_index=True)
weather_df["datetime_utc"] = weather_df["datetime_utc"].dt.tz_localize(None)

print("\nDone! Weather data shape:", weather_df.shape)
weather_df.head()

Fetching Seattle for BPAT...
Fetching Sacramento for CISO...
Fetching Dallas for ERCO...
Fetching Boston for ISNE...
Fetching Chicago for MISO...
Fetching New York for NYIS...
Fetching Philadelphia for PJM...
Fetching Oklahoma City for SWPP...

Done! Weather data shape: (17472, 6)


,datetime_utc,temperature,apparent_temp,humidity,precipitation,region
0,2026-03-15 00:00:00,6.85,4.705282,63.496994,0.0,BPAT
1,2026-03-15 01:00:00,7.15,4.588635,63.110184,0.0,BPAT
2,2026-03-15 02:00:00,6.40,3.482548,64.550583,0.0,BPAT
3,2026-03-15 03:00:00,5.75,2.607332,66.060349,0.0,BPAT
4,2026-03-15 04:00:00,5.35,2.095968,67.426262,0.0,BPAT


In [4]:
weather_df = weather_df.rename(columns={
    "temperature": "temperature_C",
    "apparent_temp": "apparent_temp_C",
    "humidity": "humidity_pct",
    "precipitation": "precipitation_mm"
})

weather_df.head()

,datetime_utc,temperature_C,apparent_temp_C,humidity_pct,precipitation_mm,region
0,2026-03-15 00:00:00,6.85,4.705282,63.496994,0.0,BPAT
1,2026-03-15 01:00:00,7.15,4.588635,63.110184,0.0,BPAT
2,2026-03-15 02:00:00,6.40,3.482548,64.550583,0.0,BPAT
3,2026-03-15 03:00:00,5.75,2.607332,66.060349,0.0,BPAT
4,2026-03-15 04:00:00,5.35,2.095968,67.426262,0.0,BPAT


In [5]:
merged = eia.merge(weather_df, on=["datetime_utc", "region"], how="left")

print("Merged shape:", merged.shape)
print("Missing weather values:", merged["temperature_C"].isna().sum())
merged.head()

Merged shape: (17288, 11)
Missing weather values: 0


,datetime_utc,region,demand_mwh,hour,day_of_week,month,is_weekend,temperature_C,apparent_temp_C,humidity_pct,precipitation_mm
0,2026-03-15 22:00:00,BPAT,7608,22,6,3,1,7.10,4.323064,63.326538,0.0
1,2026-03-15 23:00:00,BPAT,7638,23,6,3,1,7.10,4.528995,63.326538,0.1
2,2026-03-16 00:00:00,BPAT,7784,0,0,3,0,7.65,5.021635,66.480186,0.3
3,2026-03-16 01:00:00,BPAT,7957,1,0,3,0,7.35,4.412580,69.323738,0.4
4,2026-03-16 02:00:00,BPAT,8097,2,0,3,0,6.75,3.597219,69.448616,0.4


In [6]:
import os
os.makedirs("../data/processed", exist_ok=True)

merged.to_csv("../data/processed/eia_with_weather.csv", index=False)
print("Saved!")

Saved!


In [7]:
weather_df.to_csv("../data/raw/weather_data.csv", index=False)
print("Saved!")

Saved!
